> <p><small>This notebook is made available subject to the licence and terms set out in <a href="https://creativecommons.org/licenses/by/4.0">https://creativecommons.org/licenses/by/4.0</a>.</small></p>

<img src="https://pub-bba109a9a6ac49e3b428cdca19c34363.r2.dev/LT%20-%20Session%204.jpg">

# 4.6 AI Activity Long 1A - Lab: From Embeddings To Attention (Student)

Explore the core mechanism of transformer models, self-attention, concretely and intuitively.

60 minutes

## Overview

Attention is not just the idea that “some words matter more than others”. It is a specific computation:

1. Tokens start as embeddings.
2. Embeddings are projected into queries, keys, and values.
3. Queries are compared with keys.
4. Similarity scores are scaled.
5. Softmax converts scores into attention weights.
6. Attention weights combine values into new representations.

Later in the notebook, you will also explore causal masking and a simple two-head example.

---
> ℹ️ **Info:**  
> The vectors and projection matrices in this notebook are deliberately tiny. The goal is not to build a production transformer, but to make each step of attention visible and understandable.

### What you'll learn:

- Explain attention as weighted aggregation.
- Explain masking as an information constraint.
- Develop intuition for multi‑head attention.


> 💻 **Your tasks:**  
> Complete the notebook by filling in the scaffolded code cells.
> By the end of the lab, you should be able to:
> 1. Construct queries, keys, and values from token embeddings.
> 2. Compute attention scores using dot products.
> 3. Apply scaling and softmax.
> 4. Visualize attention weights as a heatmap.
> 5. Apply causal masking.
> 6. Compare two simple attention heads.

---
> 💭 **Reflection:**  
> Keep a note of how each operation changes the attention pattern. The aim is to explain the computation, not just produce the final array.

## Step 1 — Start from token embeddings

You will work with a short 4-token sequence:

- **the**
- **smart**
- **student**
- **studies**

Each token is represented by a small vector. In a real transformer, these vectors are learned. Here, the vectors are small so that you can inspect them directly.

> 💭 **Reflection:**
> Before running the code, which tokens do you expect to have similar representations? Which token relationships might matter most?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tokens = ["the", "smart", "student", "studies"]

X = np.array([
    [1, 0, 1],
    [1, 2, 0],
    [2, 1, 1],
    [0, 1, 2],
], dtype=float)

X

## Step 2 — Define queries, keys, and values

Attention usually uses three projected versions of the embeddings:

- **Query (Q):** what a token is looking for
- **Key (K):** what a token offers to be matched against
- **Value (V):** what information a token contributes

In a real model, these come from learned projection matrices. In this lab, identity matrices are used first so that the arithmetic stays simple.

In [ ]:
# Add your code here.

# Define identity projection matrices.
# Hint: use np.eye(3)
W_Q =
W_K =
W_V =

# Compute Q, K, and V.
Q =
K =
V =

Q, K, V

## Step 3 — Compute similarity scores

The next step is to compare each query with every key.

This produces a matrix of similarity scores:

```
[
  text{scores} = QK^T
]
```

A large score means a strong match. A small score means a weak match.

In [ ]:
# Add your code here.

# Compute the similarity score matrix.
# Hint: use matrix multiplication and K.T
scores =

scores

## Step 4 — Scale the scores

Transformers scale the scores by \(\sqrt{d_k}\), where \(d_k\) is the key dimension.

> ℹ️ **Info:**  
> Without scaling, high-dimensional dot products can become large. Large values passed into softmax can produce overly sharp attention distributions.

In [ ]:
# Add your code here.

# Compute d_k and scale the scores by sqrt(d_k).
d_k =
scaled_scores =

scaled_scores

## Step 5 — Apply softmax

Softmax turns raw scores into attention weights.

After softmax:

- all weights are non-negative
- each row sums to 1

Each row now describes how strongly one token attends to the others.

In [ ]:
# Add your code here.

def softmax(x):
    """Compute row-wise softmax for a matrix of scores.

    Args:
        x: A 2D NumPy array of attention scores.

    Returns:
        A 2D NumPy array where each row sums to 1.
    """
    # Add your code here.


# Compute attention weights.
weights =

weights

## Step 6 — Visualize the attention weights

A heatmap makes attention patterns easier to interpret than raw numbers.

> ℹ️ **Info:**  
> Darker or brighter cells indicate stronger attention, depending on the colour scale. Each row corresponds to the token doing the attending, and each column corresponds to the token being attended to.

In [ ]:
def plot_attention(weights, tokens, title):
    """Display an attention matrix as a heatmap.

    Args:
        weights: A 2D NumPy array of attention weights.
        tokens: Token labels used on both axes.
        title: Title shown above the heatmap.
    """
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(weights)

    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_yticklabels(tokens)

    ax.set_xlabel("Attended-to token")
    ax.set_ylabel("Attending token")
    ax.set_title(title)

    for i in range(weights.shape[0]):
        for j in range(weights.shape[1]):
            ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center")

    fig.colorbar(im, ax=ax)
    plt.show()

plot_attention(weights, tokens, "Head 1 attention weights")

## Step 7 — Compute the new token representations

Once you have the attention weights, use them to combine the Value vectors:

```
[
  text{output} = \text{weights} \times V
]
```

This is the core of attention: each token receives a new representation by combining information from other tokens.

In [ ]:
# Add your code here.

# Compute the new token representations.
output =

output

## Step 8 — Add causal masking

In language generation, tokens should not look into the future.

That means:

- token 1 cannot attend to tokens 2, 3, or 4
- token 2 cannot attend to tokens 3 or 4
- and so on

Before softmax, forbidden future positions are set to a very large negative number. After softmax, those positions receive attention weight 0.

In [ ]:
# Add your code here.

# Create a causal mask.
# Hint: np.triu(np.ones_like(scores), k=1) creates 1s above the diagonal.
mask =

# Apply the mask to the scaled scores.
masked_scores = scaled_scores.copy()
masked_scores[mask == 1] =

# Recompute masked weights and masked output.
masked_weights =
masked_output =

masked_weights, masked_output

In [ ]:
# Visualize the masked attention weights.
plot_attention(masked_weights, tokens, "Causal masked attention weights")

## Step 9 — Add a second attention head

Multi-head attention means using several attention mechanisms in parallel.

Each head has different projection matrices, so each head can focus on different relationships.

This step uses a simple second head that ignores one embedding dimension. This is not a full transformer implementation, but it shows how changing projections changes the attention pattern.

In [ ]:
# Add your code here.

# Define a second set of projection matrices.
# Hint: this matrix keeps the first two dimensions and removes the third.
W_Q2 = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 0],
], dtype=float)

W_K2 =
W_V2 =

# Compute Q2, K2, and V2.
Q2 =
K2 =
V2 =

# Compute scores, scaled scores, weights, and output for Head 2.
scores2 =
scaled_scores2 =
weights2 =
output2 =

weights2, output2

In [ ]:
# Visualize the second head.
plot_attention(weights2, tokens, "Head 2 attention weights")

## Step 10 — Compare the heads

The two heads should now produce different attention patterns.

> 💭 **Reflection:**  
>  Answer the questions below.
> 1. Do the two heads attend in the same way?
> 2. Which token relationships changed?
> 3. Why might it be useful for a transformer to use more than one head?
> 4. What would happen if every head learned exactly the same pattern?

In [ ]:
# Add your code here.

# Compare Head 1 and Head 2 numerically.
# Suggested output:
# print("Head 1 weights:")
# print(...)
# print("Head 2 weights:")
# print(...)


> 💭 **Reflection:**
> Write a short explanation of attention in your own words. A good explanation should answer:
> 1. Where do attention weights come from?
> 2. Why is softmax used?
> 3. What does masking do?
> 4. Why might multiple heads be useful?

> **Key goal:**  
> By the end of this notebook, you should be able to explain attention as a sequence of computations, not just as a vague intuition.